# ☁️ Topic 09 — Azure Fundamentals & Azure ML Workspace
> **Bootcamp:** Advanced ML / FAMA | Day 2 — Cloud & Scalable MLOps  
> **Target:** Master Azure cloud navigation, understand the Azure ML Workspace architecture, configure scalable compute, and manage assets with Python SDK v2.

---

### 🌟 Why Move Machine Learning to the Cloud?
On a local laptop:
* ⚠️ **Compute bottlenecks**: Training large models is constrained by local CPU/GPU and RAM limits.
* ⚠️ **Fragmented tracking**: Models and metrics live in ad-hoc local folders without governance.
* ⚠️ **Deployment friction**: Moving code from a laptop to production requires manual rewrites.

**The Azure ML Solution**: A centralized cloud workspace that provides managed auto-scaling compute, centralized data assets, automated experiment tracking via MLflow, a governed model registry, and managed zero-downtime inference endpoints.

## 🧭 1. Azure Core Concepts for ML Practitioners

Before diving into Azure ML, let's understand how Azure organizes cloud resources:

| Azure Concept | ML Analogy | Role in MLOps |
|---|---|---|
| **Subscription** | Master Billing Account | Boundary for financial budgets, cost tracking, and team quotas. |
| **Resource Group** | Project Namespace | Logical folder grouping all interrelated resources (Workspace, Storage, Key Vault). |
| **Region** | Physical Data Center | Geographic location (e.g. `East US`, `Southeast Asia`) chosen for compliance and latency. |
| **Storage Account** | Data Lake / Blob Store | Stores training datasets, experiment artifacts, and serialized model files (`.pkl`). |
| **Key Vault** | Secrets Locker | Safely stores database passwords, access tokens, and API keys. |
| **Container Registry (ACR)**| Docker Image Hub | Stores custom Docker images containing specialized ML dependencies. |
| **Application Insights** | Telemetry Watchtower | Captures real-time latency, request rates, error logs, and model performance.

## 🏢 2. The Azure Machine Learning Workspace

The **Azure ML Workspace** is the top-level foundational resource for all AI/ML activities. It acts as the central control plane connecting your data, compute clusters, experiments, and deployed models.

```
Azure ML Workspace
├── 🖥️ Compute
│   ├── Compute Instances (Interactive Jupyter / EDA)
│   ├── Compute Clusters (Auto-scaling multi-node training: 0 → N nodes)
│   └── Serverless Compute (Instant on-demand jobs)
├── 💾 Data
│   ├── Datastores (Secure links to Blob / ADLS Gen2)
│   └── Data Assets (Versioned dataset snapshots with lineage)
├── 🔬 Jobs
│   ├── Command Jobs (Single script training execution)
│   ├── Sweep Jobs (Automated hyperparameter search)
│   └── Pipeline Jobs (Multi-step end-to-end DAG workflows)
├── 📦 Models
│   └── Model Registry (Versioned models with staging & lineage)
├── 🐳 Environments
│   └── Curated & Custom Docker + Conda runtime environments
└── 🚀 Endpoints
    ├── Managed Online Endpoints (Real-time REST inference)
    └── Batch Endpoints (High-throughput bulk scoring)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set visual style
fig, ax = plt.subplots(figsize=(14, 8), dpi=120)
ax.set_facecolor('#F8FAFC')
fig.patch.set_facecolor('#FFFFFF')

# Outer Workspace Frame
ws_box = patches.FancyBboxPatch((0.5, 0.5), 13.0, 7.0, boxstyle="round,pad=0.2",
                                facecolor='#FFFFFF', edgecolor='#2563EB', linewidth=2.5)
ax.add_patch(ws_box)

# Workspace Header
plt.text(1.0, 7.1, "☁️ Azure ML Workspace (Central Control Plane)", fontsize=16, fontweight='bold', color='#1E3A8A')
plt.text(1.0, 6.7, "Unified management for assets, compute, tracking, and deployment", fontsize=11, color='#64748B')

# Component Categories (3 Columns)
categories = [
    {
        "title": "🖥️ COMPUTE & INFRA",
        "color": "#EFF6FF", "border": "#3B82F6", "text_c": "#1D4ED8", "x": 0.9,
        "items": [
            ("Compute Instances", "Interactive Jupyter notebooks"),
            ("Compute Clusters", "Auto-scale (0 -> N) for training"),
            ("Serverless Compute", "On-demand execution with zero setup"),
            ("Kubernetes (AKS)", "High-scale enterprise deployment")
        ]
    },
    {
        "title": "📦 ASSETS & EXPERIMENTS",
        "color": "#ECFDF5", "border": "#10B981", "text_c": "#065F46", "x": 5.0,
        "items": [
            ("Datastores & Assets", "Versioned datasets & storage pointers"),
            ("Environments", "Docker images & Conda specs"),
            ("Jobs & Pipelines", "MLflow-tracked runs & DAGs"),
            ("Model Registry", "Versioned models (Dev -> Staging -> Prod)")
        ]
    },
    {
        "title": "🚀 SERVING & OPS",
        "color": "#FEF3C7", "border": "#F59E0B", "text_c": "#92400E", "x": 9.1,
        "items": [
            ("Online Endpoints", "Low-latency REST APIs (Blue/Green)"),
            ("Batch Endpoints", "High-throughput asynchronous scoring"),
            ("App Insights", "Live latency & error telemetry"),
            ("Governance & RBAC", "Enterprise security & audit logs")
        ]
    }
]

for cat in categories:
    cx = cat["x"]
    card = patches.FancyBboxPatch((cx, 1.0), 3.8, 5.3, boxstyle="round,pad=0.15",
                                 facecolor=cat["color"], edgecolor=cat["border"], linewidth=1.5)
    ax.add_patch(card)
    plt.text(cx + 0.2, 5.9, cat["title"], fontsize=12, fontweight='bold', color=cat["text_c"])
    
    y_pos = 5.2
    for name, desc in cat["items"]:
        item_box = patches.FancyBboxPatch((cx + 0.15, y_pos - 0.7), 3.5, 0.75, boxstyle="round,pad=0.08",
                                         facecolor='#FFFFFF', edgecolor='#E2E8F0', linewidth=1)
        ax.add_patch(item_box)
        plt.text(cx + 0.3, y_pos - 0.25, f"• {name}", fontsize=10, fontweight='bold', color='#0F172A')
        plt.text(cx + 0.5, y_pos - 0.55, desc, fontsize=8.5, color='#64748B')
        y_pos -= 1.05

ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
plt.title("Azure Machine Learning Architecture Map", fontsize=16, pad=15, fontweight='bold', color='#0F172A')
plt.tight_layout()
plt.show()

## 💻 3. Azure ML Compute Options & Cost Optimization

Understanding when to use each compute target is essential for managing cloud budgets:

| Compute Target | Scaling Behavior | Billed When | Recommended Workload |
|---|---|---|---|
| **Compute Instance** | Single dedicated VM (no auto-scale) | Whenever VM state is `Running` | Interactive EDA, debugging, Jupyter notebooks |
| **Compute Cluster** | Auto-scales **0 → N nodes** | **Only during active job execution** | Production training jobs, hyperparameter sweeps |
| **Serverless Compute** | Managed dynamic provisioning | Exact seconds of execution | Quick ad-hoc training scripts with zero cluster management |
| **Managed Online Endpoint** | Auto-scales based on traffic/CPU | 24/7 per provisioned replica | Real-time REST API serving (low latency) |
| **Batch Endpoint** | Automatically starts & stops compute | Only while scoring dataset | Nightly batch scoring jobs |

> 💰 **MLOps Cost Optimization Pro-Tip**:
> Set `min_instances = 0` and `idle_time_before_scale_down = 120` seconds on your training compute clusters. When training finishes, nodes automatically deprovision to **0**, reducing idle cloud costs to **$0.00**!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(14, 7), dpi=120)
ax.set_facecolor('#F8FAFC')
fig.patch.set_facecolor('#FFFFFF')

# Resource Group Box
rg_box = patches.FancyBboxPatch((0.5, 0.5), 13.0, 6.0, boxstyle="round,pad=0.2",
                               facecolor='#FFFFFF', edgecolor='#64748B', linewidth=2, linestyle='--')
ax.add_patch(rg_box)
plt.text(0.8, 6.2, "📁 Azure Resource Group (Project Namespace)", fontsize=13, fontweight='bold', color='#334155')

# Central AML Workspace
aml_box = patches.FancyBboxPatch((4.5, 2.0), 5.0, 3.2, boxstyle="round,pad=0.2",
                                facecolor='#EFF6FF', edgecolor='#2563EB', linewidth=2.5)
ax.add_patch(aml_box)
plt.text(4.8, 4.7, "☁️ Azure ML Workspace", fontsize=14, fontweight='bold', color='#1D4ED8')
plt.text(4.8, 4.2, "• Experiment Orchestration\n• Compute Cluster Management\n• Model Staging & Lineage\n• Endpoint Routing", fontsize=10, color='#1E3A8A')

# Companion Services around AML
companions = [
    ("💾 Azure Blob Storage", "Default Datastore\nDatasets, Outputs, Logs", 1.0, 3.8, '#ECFDF5', '#10B981', '#065F46'),
    ("🔐 Azure Key Vault", "Secrets & Credentials\nDB Strings, API Keys", 1.0, 1.0, '#FEF3C7', '#F59E0B', '#92400E'),
    ("🐳 Container Registry (ACR)", "Docker Image Store\nCustom ML Environments", 10.0, 3.8, '#F3E8FF', '#A855F7', '#6B21A8'),
    ("📊 Application Insights", "Live Telemetry\nLatency, QPS, Error Rates", 10.0, 1.0, '#FFE4E6', '#F43F5E', '#9F1239')
]

for title, desc, cx, cy, bg, border, tc in companions:
    box = patches.FancyBboxPatch((cx, cy), 2.8, 1.8, boxstyle="round,pad=0.15",
                                facecolor=bg, edgecolor=border, linewidth=1.8)
    ax.add_patch(box)
    plt.text(cx + 0.15, cy + 1.4, title, fontsize=10.5, fontweight='bold', color=tc)
    plt.text(cx + 0.15, cy + 0.6, desc, fontsize=8.5, color='#475569')

# Connective arrows
arrow_style = dict(arrowstyle="<->", color="#94A3B8", lw=2, mutation_scale=15)
ax.annotate("", xy=(3.9, 4.7), xytext=(4.4, 4.0), arrowprops=arrow_style)
ax.annotate("", xy=(3.9, 2.0), xytext=(4.4, 2.8), arrowprops=arrow_style)
ax.annotate("", xy=(9.9, 4.7), xytext=(9.6, 4.0), arrowprops=arrow_style)
ax.annotate("", xy=(9.9, 2.0), xytext=(9.6, 2.8), arrowprops=arrow_style)

ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
plt.title("Azure ML Workspace & Companion Cloud Infrastructure", fontsize=15, pad=15, fontweight='bold', color='#0F172A')
plt.tight_layout()
plt.show()

## 🐍 4. Connecting via Azure ML Python SDK v2

Azure ML SDK v2 provides a streamlined, object-oriented interface for interacting with cloud resources.

### Authentication Strategy: `DefaultAzureCredential`
`DefaultAzureCredential` from `azure-identity` tries authentication mechanisms in the following priority:
1. **Environment Variables** (used in CI/CD pipelines via Service Principals: `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_TENANT_ID`)
2. **Managed Identity** (used when running inside Azure VMs / Compute Instances)
3. **Azure CLI** (`az login` session on local developer machine)
4. **Interactive Browser** (prompts a web browser login modal)

```python
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Authenticate automatically
credential = DefaultAzureCredential()

# Initialize MLClient
ml_client = MLClient(
    credential=credential,
    subscription_id="<YOUR_SUBSCRIPTION_ID>",
    resource_group_name="<YOUR_RESOURCE_GROUP>",
    workspace_name="<YOUR_WORKSPACE_NAME>"
)
```

In [ ]:
import os
import sys

print("=" * 70)
print("🔍 AZURE ML SDK v2 WORKSPACE EXPLORER")
print("=" * 70)

# Check if Azure SDK is available and if live credentials are configured
AZURE_SDK_AVAILABLE = False
try:
    from azure.ai.ml import MLClient
    from azure.identity import DefaultAzureCredential
    AZURE_SDK_AVAILABLE = True
except ImportError:
    AZURE_SDK_AVAILABLE = False

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "sub-fama-prod-001")
resource_group = os.getenv("AZURE_RESOURCE_GROUP", "rg-fama-mlops-eastus")
workspace_name = os.getenv("AZURE_WORKSPACE_NAME", "aml-fama-workspace")

connected_live = False

if AZURE_SDK_AVAILABLE and os.getenv("AZURE_CLIENT_ID"):
    try:
        credential = DefaultAzureCredential()
        ml_client = MLClient(credential, subscription_id, resource_group, workspace_name)
        ws = ml_client.workspaces.get(workspace_name)
        print(f"✅ Successfully connected to Live Workspace: {ws.name}")
        print(f"📍 Region / Location: {ws.location}")
        connected_live = True
    except Exception as e:
        print(f"ℹ️ Live Azure connection skipped ({e}). Running in offline simulation mode.")
        connected_live = False

if not connected_live:
    print(f"🌐 [SIMULATED WORKSPACE SESSION]")
    print(f"• Subscription ID : {subscription_id}")
    print(f"• Resource Group  : {resource_group}")
    print(f"• Workspace Name  : {workspace_name}")
    print(f"• Region          : East US (Primary)")
    print(f"• Status          : Succeeded (Active)")
    print("-" * 70)
    
    # 1. Compute Inventory
    print("\n🖥️  COMPUTE TARGETS:")
    mock_compute = [
        {"name": "ci-notebook-dev", "type": "ComputeInstance", "vm_size": "Standard_DS3_v2", "state": "Running", "nodes": "1"},
        {"name": "cpu-cluster-train", "type": "AmlCompute", "vm_size": "Standard_DS12_v2", "state": "Idle", "nodes": "0-4 (Auto-scale)"},
        {"name": "gpu-cluster-fast", "type": "AmlCompute", "vm_size": "Standard_NC6s_v3", "state": "Idle", "nodes": "0-2 (GPU Scale)"},
    ]
    for c in mock_compute:
        print(f"  • {c['name']:<20} | Type: {c['type']:<16} | Size: {c['vm_size']:<16} | Nodes: {c['nodes']:<18} | State: {c['state']}")
        
    # 2. Datastores Inventory
    print("\n💾 DATASTORES:")
    mock_datastores = [
        {"name": "workspaceblobstore", "type": "AzureBlob", "default": True, "container": "azureml-blobstore"},
        {"name": "workspaceworkingdir", "type": "AzureBlob", "default": False, "container": "azureml-workingdir"},
        {"name": "raw_customer_lake", "type": "AzureDataLakeGen2", "default": False, "container": "fama-data-lake"},
    ]
    for ds in mock_datastores:
        def_tag = " (Default)" if ds["default"] else ""
        print(f"  • {ds['name']:<20} | Type: {ds['type']:<18} | Container: {ds['container']}{def_tag}")
        
    # 3. Model Registry Inventory
    print("\n📦 REGISTERED MODELS:")
    mock_models = [
        {"name": "fama_risk_xgboost", "version": "3", "stage": "Production", "framework": "Scikit-Learn / XGBoost", "accuracy": "0.942"},
        {"name": "fama_fraud_lightgbm", "version": "1", "stage": "Staging", "framework": "LightGBM", "accuracy": "0.918"},
        {"name": "fama_churn_predictor", "version": "2", "stage": "Development", "framework": "PyTorch", "accuracy": "0.885"},
    ]
    for m in mock_models:
        print(f"  • {m['name']:<22} | v{m['version']} | Stage: {m['stage']:<12} | Framework: {m['framework']:<22} | Metric: {m['accuracy']}")

print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate 24-hour training workload cost comparison
hours = np.arange(0, 25)

# Workload: Training runs for 4 hours (hours 8 to 12)
training_active = np.zeros(25)
training_active[8:12] = 4 # 4 nodes active

# Hourly rate per node (e.g. $0.60/hr for Standard_DS12_v2)
hourly_rate = 0.60

# 1. Compute Instance (Fixed running 24/7)
cost_instance = np.cumsum(np.ones(25) * 1 * hourly_rate)

# 2. Compute Cluster (Scale-to-zero: only charges when training is active)
cost_cluster = np.cumsum(training_active * hourly_rate)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), dpi=120)
fig.patch.set_facecolor('#FFFFFF')

# Plot 1: Node scaling profile
ax1.step(hours, training_active, where='mid', color='#2563EB', linewidth=2.5, label='Compute Cluster (Auto-scale)')
ax1.plot(hours, np.ones(25), color='#EF4444', linestyle='--', linewidth=2, label='Compute Instance (Fixed 1 node)')
ax1.set_title("Active Compute Nodes Over 24 Hours", fontsize=13, fontweight='bold', color='#0F172A')
ax1.set_xlabel("Hour of Day", fontsize=11)
ax1.set_ylabel("Active Node Count", fontsize=11)
ax1.set_ylim(-0.5, 5)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right', frameon=True)
ax1.set_facecolor('#F8FAFC')

# Plot 2: Cumulative Cost Comparison
ax2.plot(hours, cost_instance, color='#EF4444', linewidth=2.5, label=f'Compute Instance: ${cost_instance[-1]:.2f}')
ax2.plot(hours, cost_cluster, color='#10B981', linewidth=2.5, label=f'Auto-scale Cluster (0-4): ${cost_cluster[-1]:.2f}')
ax2.fill_between(hours, cost_cluster, cost_instance, color='#10B981', alpha=0.15, label=f'Savings: ${(cost_instance[-1] - cost_cluster[-1]):.2f} (33% cheaper)')
ax2.set_title("Cumulative Cloud Cost Comparison", fontsize=13, fontweight='bold', color='#0F172A')
ax2.set_xlabel("Hour of Day", fontsize=11)
ax2.set_ylabel("Cumulative Cost (USD)", fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='lower right', frameon=True)
ax2.set_facecolor('#F8FAFC')

plt.suptitle("Azure ML Compute Cost Optimization: Auto-Scale to Zero", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

features = [
    'Provisioning Speed',
    'Elastic Auto-Scaling',
    'Experiment Tracking',
    'Model Governance',
    'Security & RBAC',
    'Zero-Downtime Serving'
]

on_prem_scores = [2, 3, 4, 3, 5, 4]
azure_scores = [9, 10, 9, 9, 10, 9]

y_pos = np.arange(len(features))

fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
fig.patch.set_facecolor('#FFFFFF')
ax.set_facecolor('#F8FAFC')

bar_height = 0.35
rects1 = ax.barh(y_pos - bar_height/2, on_prem_scores, bar_height, label='On-Premise Infrastructure', color='#94A3B8')
rects2 = ax.barh(y_pos + bar_height/2, azure_scores, bar_height, label='Azure ML Enterprise Platform', color='#2563EB')

ax.set_xlabel('Capability & Efficiency Score (1 - 10)', fontsize=12, fontweight='bold', color='#0F172A')
ax.set_title('On-Premise vs. Azure ML Platform Comparison', fontsize=15, fontweight='bold', pad=15, color='#0F172A')
ax.set_yticks(y_pos)
ax.set_yticklabels(features, fontsize=11, fontweight='bold', color='#334155')
ax.set_xlim(0, 11)
ax.grid(axis='x', linestyle=':', alpha=0.6)
ax.legend(loc='lower right', frameon=True, fontsize=11)

# Add score labels
for rect in rects1:
    width = rect.get_width()
    ax.annotate(f'{width}/10', xy=(width + 0.2, rect.get_y() + rect.get_height()/2),
                va='center', ha='left', fontsize=9.5, color='#64748B')
for rect in rects2:
    width = rect.get_width()
    ax.annotate(f'{width}/10', xy=(width + 0.2, rect.get_y() + rect.get_height()/2),
                va='center', ha='left', fontsize=9.5, fontweight='bold', color='#1D4ED8')

plt.tight_layout()
plt.show()

## 📝 5. Key Takeaways & Knowledge Check

### 🔑 Essential Rules of Azure ML
1. **Workspace is the Single Pane of Glass**: Centralizes data, compute, experiment history, and model versions for complete traceability and audit compliance.
2. **Auto-Scale to Zero is King**: Always set `min_instances = 0` on training compute clusters so you only pay for compute when jobs are actively running.
3. **Four Companion Resources**: When provisioning a Workspace, Azure automatically creates an **Azure Storage Account**, **Key Vault**, **Container Registry**, and **Application Insights**.
4. **SDK v2 is Declarative**: Use clean, modern Python objects and YAML definitions for all MLOps automation and CI/CD pipelines.

---

### 🏋️ Hands-On Exercises for Learners

1. **Exercise 1 (Compute Cluster Config)**: Write an Azure ML SDK v2 snippet using `AmlCompute` to define a cluster named `gpu-cluster-large` with `Standard_NC12s_v3`, `min_instances = 0`, `max_instances = 4`, and `idle_time_before_scale_down = 180`.
2. **Exercise 2 (Datastore Registration)**: Identify the difference between a Datastore (connection credentials to storage) and a Data Asset (versioned reference to specific parquet/CSV files).

---

### ❓ Self-Check Quiz

1. **Q1: Which Azure service is used to store sensitive API keys and database connection strings used during ML training?**  
   *A: Azure Key Vault.*

2. **Q2: Why should you avoid using a Compute Instance to run a 10-hour distributed hyperparameter tuning job?**  
   *A: A Compute Instance is a single dedicated VM that does not scale out to parallel nodes and incurs costs 24/7 if not manually stopped.*

3. **Q3: What role does Azure Container Registry (ACR) play in Azure ML?**  
   *A: It stores the Docker images containing the exact Python runtime, libraries, and C++ packages required for training and deployment.*